In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

path_trips= "/Volumes/trips/landing/trips"
schema_path_trips = "/Volumes/trips/landing/trips/_schema_v2"  # ✅ New schema location to force fresh inference

@dp.table(
    name="trips.bronze.trips",
    comment="Raw city data from CSV files",
    table_properties={
        "quality": "bronze"
    }
)
def raw_city_bronze():
    df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("cloudFiles.schemaLocation", schema_path_trips)  # ✅
            .option("cloudFiles.includeExistingFiles", "true")
            .load(path_trips)
    )
    # Rename columns with invalid characters (parentheses not allowed in Delta)
    df = df.withColumnRenamed("distance_travelled(km)", "distance_travelled_km")
    return df.withColumn("ingestion_date", F.current_timestamp())